<a href="https://colab.research.google.com/github/abbbymuaa/mssp6070/blob/main/WeeklyModules/Week10/Assignment_2_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 — Improved Internet Sales Analysis

**Purpose:** Refactor the original notebook while preserving the four questions and additional year-by-category chart. Improvements focus on portable loading, data validation, transparent product classification, customer-level comparisons, and consistent plots.

**Input:** `InternetSales.csv` (the original notebook references `mssp6070/data/InternetSales.csv`). This notebook does **not** bundle the CSV. Place it next to the notebook, under `data/`, or in `mssp6070/data/`; see the loading cell below.

## 1. Setup and load the dataset

Only imports used in the analysis are included. Unlike the original notebook, these cells do not change the working directory or automatically clone a repository.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams.update({"figure.figsize": (10, 5.5), "axes.grid": True, "grid.alpha": 0.2})


In [ ]:
# Optional: set DATA_PATH to your full CSV path, e.g. Path("/content/mssp6070/data/InternetSales.csv")
DATA_PATH = None

candidates = [
    Path("InternetSales.csv"),
    Path("data/InternetSales.csv"),
    Path("mssp6070/data/InternetSales.csv"),
    Path("/content/mssp6070/data/InternetSales.csv"),
]

if DATA_PATH is not None:
    candidates.insert(0, Path(DATA_PATH))

csv_path = next((p for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(
        "InternetSales.csv not found. Place it next to this notebook or set DATA_PATH "
        "to its full location. Checked: " + ", ".join(str(p) for p in candidates)
    )

# Latin-1 preserves compatibility with the original notebook's encoding.
df = pd.read_csv(csv_path, encoding="latin1")
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns from {csv_path}")


## 2. Validate and prepare data

Fail early if expected columns are absent. Convert dates and numbers deliberately, and report missingness without silently dropping rows.

In [ ]:
required_columns = [
    "OrderDate", "EnglishCountryRegionName", "SalesAmount",
    "CustomerAlternateKey", "YearlyIncome", "Gender",
    "EnglishProductName", "UnitPrice", "HouseOwnerFlag"
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

for column in ["SalesAmount", "YearlyIncome", "UnitPrice"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["Year"] = df["OrderDate"].dt.year

print("Dataset shape:", df.shape)
print("Missing values in analysis columns:")
display(df[required_columns + ["Year"]].isna().sum().to_frame("Missing count"))
print("Sales amount summary:")
display(df["SalesAmount"].describe().to_frame("SalesAmount"))


In [ ]:
# Basic checks that are helpful before interpreting results.
assert df["SalesAmount"].notna().any(), "SalesAmount has no numeric values."
assert df["Year"].notna().any(), "OrderDate has no valid dates."
print("Rows with invalid/missing order dates:", int(df["Year"].isna().sum()))
print("Rows with missing customer IDs:", int(df["CustomerAlternateKey"].isna().sum()))
print("Negative sales transactions:", int((df["SalesAmount"] < 0).sum()))


## 3. Q1 — Total Internet sales by year and country

Sum transaction sales (not averages). Rows with unknown years or countries are excluded from the country-year breakdown; check the missing-value report above.

In [ ]:
annual_country_sales = (
    df.dropna(subset=["Year", "EnglishCountryRegionName", "SalesAmount"])
      .groupby(["Year", "EnglishCountryRegionName"], as_index=False)["SalesAmount"]
      .sum()
)

sales_by_country_year = annual_country_sales.pivot(
    index="EnglishCountryRegionName", columns="Year", values="SalesAmount"
).fillna(0)
print("Internet sales by country and year ($):")
display(sales_by_country_year.round(2))

ax = sales_by_country_year.T.plot(kind="bar", figsize=(11, 6))
ax.set(title="Internet Sales by Year and Country", xlabel="Order year", ylabel="Total sales ($)")
ax.legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 4. Q2 — Customer demographics by country

The original file contains transaction rows, not one row per customer. We first check whether customer attributes disagree across rows, then create one record per nonmissing customer ID. If disagreements exist, the first recorded value is retained; investigate those IDs before drawing firm demographic conclusions.

In [ ]:
customer_fields = ["EnglishCountryRegionName", "YearlyIncome", "Gender", "HouseOwnerFlag"]
customer_rows = df.dropna(subset=["CustomerAlternateKey"]).copy()
conflict_counts = {}
for field in customer_fields:
    conflict_counts[field] = int((
        customer_rows.groupby("CustomerAlternateKey")[field].nunique(dropna=True) > 1
    ).sum())
print("Number of customer IDs with conflicting values across transactions:")
display(pd.Series(conflict_counts, name="Conflicting customer IDs").to_frame())

customers = customer_rows.drop_duplicates(subset="CustomerAlternateKey").copy()
print(f"Unique customers: {len(customers):,}")


In [ ]:
income_stats = (
    customers.groupby("EnglishCountryRegionName")["YearlyIncome"]
             .agg(count="count", mean="mean", min="min", max="max", std="std")
             .sort_values("mean", ascending=False)
)
print("Yearly income statistics by country ($):")
display(income_stats.round(2))

gender_counts = pd.crosstab(customers["EnglishCountryRegionName"], customers["Gender"])
gender_percent = pd.crosstab(
    customers["EnglishCountryRegionName"], customers["Gender"], normalize="index"
).mul(100)
print("Gender counts:")
display(gender_counts)
print("Gender percentages within each country (%):")
display(gender_percent.round(1))

ax = income_stats["mean"].plot(kind="bar", color="steelblue")
ax.set(title="Average Customer Yearly Income by Country", xlabel="Country", ylabel="Average yearly income ($)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 5. Q3 — Product-category sales

**Classification note:** Use a pre-existing category column if available. Otherwise, apply a transparent name-based heuristic. A name-based rule is not a reliable substitute for the official Adventure Works product dimension, so we preserve an **Unclassified** bucket rather than calling all unmatched products accessories. Verify category mapping against the dataset/codebook before treating these totals as definitive.

In [ ]:
category_candidates = ["EnglishProductCategoryName", "ProductCategoryName", "Category"]
source_category_column = next((c for c in category_candidates if c in df.columns), None)

if source_category_column is not None:
    df["ProductCategory"] = df[source_category_column].astype("string").str.strip().fillna("Unclassified")
    print(f"Using existing category column: {source_category_column}")
else:
    def categorize_product(product_name):
        if pd.isna(product_name):
            return "Unclassified"
        name = str(product_name).lower()
        if name.startswith(("mountain-", "road-", "touring-")):
            return "Bikes"
        if any(word in name for word in ("jersey", "shorts", "gloves", "socks", "vest", "cap")):
            return "Clothing"
        if any(word in name for word in (
            "helmet", "bottle", "cage", "tire", "tube", "pump", "lock",
            "light", "fender", "rack", "pedal", "stand", "bag", "cleaner", "patch"
        )):
            return "Accessories"
        return "Unclassified"

    df["ProductCategory"] = df["EnglishProductName"].apply(categorize_product)
    print("Using approximate name-based product classification (no official category column found).")

print("Category frequency by transaction:")
display(df["ProductCategory"].value_counts(dropna=False).to_frame("Transactions"))


In [ ]:
category_sales = (
    df.dropna(subset=["SalesAmount"])
      .groupby("ProductCategory")["SalesAmount"]
      .sum()
      .sort_values(ascending=False)
)
print("Total sales by product category ($):")
display(category_sales.round(2).to_frame("Total sales ($)"))

classified_sales = category_sales.drop(labels="Unclassified", errors="ignore")
if classified_sales.empty:
    print("No reliably classified category found; top category and average price not computed.")
else:
    top_category = classified_sales.idxmax()
    top_category_prices = df.loc[df["ProductCategory"] == top_category, "UnitPrice"]
    print(f"Highest-sales classified category: {top_category}")
    print(f"Mean transaction-row UnitPrice for {top_category}: ${top_category_prices.mean():,.2f}")
    print("Note: this is a transaction-row mean, not a unique-product average or quantity-weighted price.")

ax = category_sales.plot(kind="bar", color="steelblue")
ax.set(title="Total Internet Sales by Product Category", xlabel="Product category", ylabel="Total sales ($)")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## 6. Q4 — Spending by homeownership status

Transaction-level sales and lifetime spending per customer answer different questions. This section shows both, and the chart compares **customer-level spending** so repeat purchasers are not overrepresented as separate observations.

In [ ]:
# Normalize the most common numeric/string forms of homeownership status.
owner_labels = {0: "Non-Homeowner", 1: "Homeowner", "0": "Non-Homeowner", "1": "Homeowner"}
df["Homeownership"] = df["HouseOwnerFlag"].map(owner_labels).fillna("Unknown")

home_sales = (
    df.groupby("Homeownership")["SalesAmount"]
      .agg(total_sales="sum", transaction_rows="count", mean_transaction_sales="mean")
)
print("Transaction-level sales by homeownership:")
display(home_sales.round(2))

# Group by customer and status; customers with conflicting statuses are flagged above.
customer_spending = (
    df.dropna(subset=["CustomerAlternateKey", "SalesAmount"])
      .groupby(["CustomerAlternateKey", "Homeownership"], as_index=False)["SalesAmount"]
      .sum()
)

home_comparison = (
    customer_spending.groupby("Homeownership")["SalesAmount"]
      .agg(customers="count", mean_customer_spending="mean", median_customer_spending="median", std="std")
)
print("Customer-level spending by homeownership:")
display(home_comparison.round(2))

ax = home_comparison["mean_customer_spending"].plot(kind="bar", color="steelblue")
ax.set(title="Average Total Spending per Customer by Homeownership", xlabel="Homeownership", ylabel="Average customer spending ($)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 7. Additional view — Product categories by year

A stacked bar chart shows how each category contributes to annual sales. If the fallback product-name classification was used, interpret it cautiously.

In [ ]:
sales_by_category_year = df.pivot_table(
    index="Year", columns="ProductCategory", values="SalesAmount", aggfunc="sum", fill_value=0
)
print("Sales by year and product category ($):")
display(sales_by_category_year.round(2))

ax = sales_by_category_year.plot(kind="bar", stacked=True, figsize=(11, 6))
ax.set(title="Annual Internet Sales by Product Category", xlabel="Order year", ylabel="Total sales ($)")
ax.legend(title="Product category", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


## 8. Takeaways and limitations

- **Q1:** Compare the country-year sales table and bar chart for changes in market contribution.
- **Q2:** Income and gender summaries represent unique customer IDs, not transactions. Review any conflicting customer attributes reported above.
- **Q3:** If an official product-category field is absent, category totals are **estimates** based on product names. Check the Unclassified group before reporting definitive category rankings.
- **Q4:** Total sales by household status and average customer spending are distinct metrics. Use the customer-level table for a customer comparison.
- **Data requirement:** Run with the original `InternetSales.csv` to generate numerical conclusions. No dataset was attached to this revised notebook, so no numbers are asserted here.
